# Causal self-attention

This notebook follows the attention implementation in `src/model/attention.py`. It inspects the score matrix, applies a causal mask, and checks that future tokens cannot change earlier outputs.

In [ ]:
import math

import torch

torch.manual_seed(7)

## Inspect scaled dot-product attention

The score tensor compares each query position with every key position.

In [ ]:
batch, time, head_dim = 1, 4, 8
q = torch.randn(batch, time, head_dim)
k = torch.randn(batch, time, head_dim)
v = torch.randn(batch, time, head_dim)

scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
future = torch.triu(torch.ones(time, time, dtype=torch.bool), diagonal=1)
masked_scores = scores.masked_fill(future, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)
output = weights @ v

print("scores:", scores.shape)
print("weights:", weights[0])
print("output:", output.shape)

Every row of the weight matrix sums to one, and entries above the diagonal are zero.

In [ ]:
assert torch.allclose(weights.sum(dim=-1), torch.ones(batch, time))
assert torch.all(weights.masked_select(future.unsqueeze(0)) == 0)
print("causal mask checks passed")

## Use the repository implementation

The reusable layer handles projection, head splitting, masking, head merging, and the output projection.

In [ ]:
from src.model.attention import CausalSelfAttention

attention = CausalSelfAttention(dim=32, n_heads=4, max_seq_len=16)
x = torch.randn(2, 8, 32)
y = attention(x)
print("input:", x.shape)
print("output:", y.shape)

In [ ]:
prefix = torch.randn(1, 4, 32)
first = attention(torch.cat([prefix, torch.zeros(1, 2, 32)], dim=1))
changed = torch.cat([prefix, torch.randn(1, 2, 32)], dim=1)
second = attention(changed)

torch.testing.assert_close(first[:, :4], second[:, :4])
print("future-token isolation check passed")